# 07 — Streaming Simulation

**Week:** 10  
**Project:** IPL MatchDay 360

Goal: Simulate controlled JSON event drops using Databricks Auto Loader / Structured Streaming with an explicit 20-field schema, watermarking, deduplication, data-quality routing, and live health metrics.


In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, BooleanType, TimestampType
)
from pyspark.sql.functions import (
    col, current_timestamp, input_file_name,
    regexp_extract, lit
)

stream_input_path = "/Volumes/workspace/default/p01_ipl_matchday_360/streaming_input/"

checkpoint_path = "/Volumes/workspace/default/p01_ipl_matchday_360/checkpoints/live_ball_event/"

print("Input path:", stream_input_path)
print("Checkpoint:", checkpoint_path)

Input path: /Volumes/workspace/default/p01_ipl_matchday_360/streaming_input/
Checkpoint: /Volumes/workspace/default/p01_ipl_matchday_360/checkpoints/live_ball_event/


In [0]:
event_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("event_ts", TimestampType(), True),
    StructField("schema_version", StringType(), True),
    StructField("match_id", StringType(), True),
    StructField("innings_no", IntegerType(), True),
    StructField("over_no", IntegerType(), True),
    StructField("ball_seq", IntegerType(), True),
    StructField("event_type", StringType(), True),
    StructField("batting_team_id", StringType(), True),
    StructField("bowling_team_id", StringType(), True),
    StructField("batter_id", StringType(), True),
    StructField("non_striker_id", StringType(), True),
    StructField("bowler_id", StringType(), True),
    StructField("runs_batter", IntegerType(), True),
    StructField("runs_extras", IntegerType(), True),
    StructField("total_runs", IntegerType(), True),
    StructField("wicket_flag", BooleanType(), True),
    StructField("dismissed_player_id", StringType(), True),
    StructField("source_delivery_id", StringType(), True),
    StructField("event_sequence", IntegerType(), True)
])

print("Number of fields:", len(event_schema.fields))

for field in event_schema.fields:
    print(field.name, "->", field.dataType)


Number of fields: 20
event_id -> StringType()
event_ts -> TimestampType()
schema_version -> StringType()
match_id -> StringType()
innings_no -> IntegerType()
over_no -> IntegerType()
ball_seq -> IntegerType()
event_type -> StringType()
batting_team_id -> StringType()
bowling_team_id -> StringType()
batter_id -> StringType()
non_striker_id -> StringType()
bowler_id -> StringType()
runs_batter -> IntegerType()
runs_extras -> IntegerType()
total_runs -> IntegerType()
wicket_flag -> BooleanType()
dismissed_player_id -> StringType()
source_delivery_id -> StringType()
event_sequence -> IntegerType()


In [0]:
stream_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .option("cloudFiles.rescuedDataColumn", "_rescued_data")
        .schema(event_schema)
        .load(stream_input_path)
        .withColumn("_source_file", input_file_name())
        .withColumn("_ingestion_ts", current_timestamp())
        .withColumn(
            "drop_id",
            regexp_extract(col("_source_file"), r"(drop_\d+)", 1)
        )
)

display(stream_df)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7772998145306630>, line 17
      1 stream_df = (
      2     spark.readStream
      3         .format("cloudFiles")
   (...)     14         )
     15 )
---> 17 display(stream_df)

File <command-7772998145306630>, line 2
      1 stream_df = (
----> 2     spark.readStream
      3         .format("cloudFiles")
      4         .option("cloudFiles.format", "json")
      5         .option("cloudFiles.schemaEvolutionMode", "rescue")
      6         .option("cloudFiles.rescuedDataColumn", "_rescued_data")
      7         .schema(event_schema)
      8         .load(stream_input_path)
      9         .withColumn("_source_file", input_file_name())
     10         .withColumn("_ingestion_ts", current_timestamp())
     11         .withColumn(
     12             "drop_id",
     13             regexp_extract(col("_source_file"), r"(drop

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.p01_ipl_matchday_360;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.p01_ipl_matchday_360;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.p01_ipl_matchday_360;

In [0]:
SHOW VOLUMES IN workspace.default;

  File <command-7772998145306635>, line 1
    SHOW VOLUMES IN workspace.default;
         ^
SyntaxError: invalid syntax


In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.default.p01_ipl_matchday_360
""")

DataFrame[]

In [0]:
spark.sql("SHOW VOLUMES IN workspace.default").show()

+--------+--------------------+
|database|         volume_name|
+--------+--------------------+
| default|             ipl_360|
| default|p01_ipl_matchday_360|
+--------+--------------------+



In [0]:
dbutils.fs.mkdirs(
    "/Volumes/workspace/default/p01_ipl_matchday_360/streaming_input/"
)

True

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/default/p01_ipl_matchday_360/"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/p01_ipl_matchday_360/streaming_input/,streaming_input/,0,1791382386969


Step 1 — Create the streaming input folder

In [0]:
dbutils.fs.mkdirs(
    "/Volumes/workspace/default/p01_ipl_matchday_360/streaming_input/"
)

print("streaming_input folder created")

streaming_input folder created


In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/default/p01_ipl_matchday_360/"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/p01_ipl_matchday_360/streaming_input/,streaming_input/,0,1791382430810


In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/default/p01_ipl_matchday_360/streaming_input/"
    )
)

[]

In [0]:
streaming_input = "/Volumes/ipl_matchday_360/default/ipl_matchday_data/streaming_input/"

dbutils.fs.mkdirs(streaming_input)

display(dbutils.fs.ls("/Volumes/ipl_matchday_360/default/ipl_matchday_data/"))

path,name,size,modificationTime
dbfs:/Volumes/ipl_matchday_360/default/ipl_matchday_data/deliveries.parquet,deliveries.parquet,2518394,1784960943000
dbfs:/Volumes/ipl_matchday_360/default/ipl_matchday_data/deliveries[1].parquet,deliveries[1].parquet,2518394,1784957702000
dbfs:/Volumes/ipl_matchday_360/default/ipl_matchday_data/live_ball_event_drop_001_sample.json,live_ball_event_drop_001_sample.json,2207,1791382936000
dbfs:/Volumes/ipl_matchday_360/default/ipl_matchday_data/matches.csv,matches.csv,85417,1784960744000
dbfs:/Volumes/ipl_matchday_360/default/ipl_matchday_data/matches[1].csv,matches[1].csv,85417,1784957702000
dbfs:/Volumes/ipl_matchday_360/default/ipl_matchday_data/players.json,players.json,64877,1784960996000
dbfs:/Volumes/ipl_matchday_360/default/ipl_matchday_data/players[1].json,players[1].json,64877,1784957702000
dbfs:/Volumes/ipl_matchday_360/default/ipl_matchday_data/streaming_input/,streaming_input/,0,1791383474385
dbfs:/Volumes/ipl_matchday_360/default/ipl_matchday_data/venues.csv,venues.csv,1442,1784961038000
dbfs:/Volumes/ipl_matchday_360/default/ipl_matchday_data/venues[1].csv,venues[1].csv,1442,1784957702000


In [0]:
dbutils.fs.cp(
    "/Volumes/ipl_matchday_360/default/ipl_matchday_data/live_ball_event_drop_001_sample.json",
    "/Volumes/ipl_matchday_360/default/ipl_matchday_data/streaming_input/live_ball_event_drop_001.json"
)

display(
    dbutils.fs.ls(
        "/Volumes/ipl_matchday_360/default/ipl_matchday_data/streaming_input/"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/ipl_matchday_360/default/ipl_matchday_data/streaming_input/live_ball_event_drop_001.json,live_ball_event_drop_001.json,2207,1791383498000


In [0]:
display(
    spark.read.json(
        "/Volumes/ipl_matchday_360/default/ipl_matchday_data/streaming_input/live_ball_event_drop_001.json"
    )
)


ball_seq,batter_id,batting_team_id,bowler_id,bowling_team_id,dismissed_player_id,event_id,event_sequence,event_ts,event_type,innings_no,match_id,non_striker_id,over_no,runs_batter,runs_extras,schema_version,source_delivery_id,total_runs,wicket_flag
1,P0140,T06,P0240,T10,null,EVT-00001,1,2026-07-20T19:00:12,boundary,1,M2022-001,P0131,1,4,0,1.0,DLV-M2022-001-1-01-01,4,false
2,P0140,T06,P0240,T10,null,EVT-00002,2,2026-07-20T19:00:24,scored_ball,1,M2022-001,P0131,1,1,0,1.0,DLV-M2022-001-1-01-02,1,false
3,P0131,T06,P0240,T10,null,EVT-00003,3,2026-07-20T19:00:36,scored_ball,1,M2022-001,P0140,1,0,0,1.0,DLV-M2022-001-1-01-03,0,false
4,P0131,T06,P0240,T10,null,EVT-00004,4,2026-07-20T19:00:48,scored_ball,1,M2022-001,P0140,1,0,0,1.0,DLV-M2022-001-1-01-04,0,false
5,P0131,T06,P0240,T10,null,EVT-00005,5,2026-07-20T19:01:00,scored_ball,1,M2022-001,P0140,1,0,0,1.0,DLV-M2022-001-1-01-05,0,false


In [0]:
import json

input_path = "/Volumes/ipl_matchday_360/default/ipl_matchday_data/streaming_input/"

drop_002 = [
    {
        "event_id": "EVT-00006",
        "event_ts": "2026-07-20T19:01:12",
        "schema_version": 1.0,
        "match_id": "M2022-001",
        "innings_no": 1,
        "over_no": 1,
        "ball_seq": 6,
        "event_type": "scored_ball",
        "batting_team_id": "T06",
        "bowling_team_id": "T10",
        "batter_id": "P0131",
        "non_striker_id": "P0140",
        "bowler_id": "P0240",
        "runs_batter": 1,
        "runs_extras": 0,
        "total_runs": 1,
        "wicket_flag": False,
        "dismissed_player_id": None,
        "source_delivery_id": "DLV-M2022-001-1-01-06",
        "event_sequence": 6
    },
    {
        "event_id": "EVT-00007",
        "event_ts": "2026-07-20T19:02:00",
        "schema_version": 1.0,
        "match_id": "M2022-001",
        "innings_no": 1,
        "over_no": 2,
        "ball_seq": 1,
        "event_type": "boundary",
        "batting_team_id": "T06",
        "bowling_team_id": "T10",
        "batter_id": "P0140",
        "non_striker_id": "P0131",
        "bowler_id": "P0240",
        "runs_batter": 4,
        "runs_extras": 0,
        "total_runs": 4,
        "wicket_flag": False,
        "dismissed_player_id": None,
        "source_delivery_id": "DLV-M2022-001-1-02-01",
        "event_sequence": 7
    },
    {
        "event_id": "EVT-00008",
        "event_ts": "2026-07-20T19:02:12",
        "schema_version": 1.0,
        "match_id": "M2022-001",
        "innings_no": 1,
        "over_no": 2,
        "ball_seq": 2,
        "event_type": "scored_ball",
        "batting_team_id": "T06",
        "bowling_team_id": "T10",
        "batter_id": "P0140",
        "non_striker_id": "P0131",
        "bowler_id": "P0240",
        "runs_batter": 2,
        "runs_extras": 0,
        "total_runs": 2,
        "wicket_flag": False,
        "dismissed_player_id": None,
        "source_delivery_id": "DLV-M2022-001-1-02-02",
        "event_sequence": 8
    }
]

with open("/tmp/drop_002.json", "w") as f:
    for event in drop_002:
        f.write(json.dumps(event) + "\n")

dbutils.fs.cp(
    "file:/tmp/drop_002.json",
    input_path + "live_ball_event_drop_002.json"
)

display(dbutils.fs.ls(input_path))

---------------------------------------------------------------------------
ExecutionError                            Traceback (most recent call last)
File <command-7772998145306649>, line 78
     75     for event in drop_002:
     76         f.write(json.dumps(event) + "\n")
---> 78 dbutils.fs.cp(
     79     "file:/tmp/drop_002.json",
     80     input_path + "live_ball_event_drop_002.json"
     81 )
     83 display(dbutils.fs.ls(input_path))

File /databricks/python_shell/lib/dbruntime/remotefshandler/RemoteFsHandler.py:57, in prettify_exception_message.<locals>.f_with_exception_handling(*args, **kwargs)
     53     pass
     55 error_exception = ExecutionError(str(e))
---> 57 raise patch_exception_with_error_details(
     58     error_exception,
     59     DriverErrorCode.REMOTE_FS_HANDLER_EXECUTION_ERROR  # type: ignore[attr-defined]
     60 ) from None

ExecutionError: (com.databricks.backend.daemon.driver.LocalFilesystemAccessDeniedException) Cannot access non /Workspace local

In [0]:
import json

input_path = "/Volumes/ipl_matchday_360/default/ipl_matchday_data/streaming_input/"

drop_002 = [
    {
        "event_id": "EVT-00006",
        "event_ts": "2026-07-20T19:01:12",
        "schema_version": 1.0,
        "match_id": "M2022-001",
        "innings_no": 1,
        "over_no": 1,
        "ball_seq": 6,
        "event_type": "scored_ball",
        "batting_team_id": "T06",
        "bowling_team_id": "T10",
        "batter_id": "P0131",
        "non_striker_id": "P0140",
        "bowler_id": "P0240",
        "runs_batter": 1,
        "runs_extras": 0,
        "total_runs": 1,
        "wicket_flag": False,
        "dismissed_player_id": None,
        "source_delivery_id": "DLV-M2022-001-1-01-06",
        "event_sequence": 6
    },
    {
        "event_id": "EVT-00007",
        "event_ts": "2026-07-20T19:02:00",
        "schema_version": 1.0,
        "match_id": "M2022-001",
        "innings_no": 1,
        "over_no": 2,
        "ball_seq": 1,
        "event_type": "boundary",
        "batting_team_id": "T06",
        "bowling_team_id": "T10",
        "batter_id": "P0140",
        "non_striker_id": "P0131",
        "bowler_id": "P0240",
        "runs_batter": 4,
        "runs_extras": 0,
        "total_runs": 4,
        "wicket_flag": False,
        "dismissed_player_id": None,
        "source_delivery_id": "DLV-M2022-001-1-02-01",
        "event_sequence": 7
    },
    {
        "event_id": "EVT-00008",
        "event_ts": "2026-07-20T19:02:12",
        "schema_version": 1.0,
        "match_id": "M2022-001",
        "innings_no": 1,
        "over_no": 2,
        "ball_seq": 2,
        "event_type": "scored_ball",
        "batting_team_id": "T06",
        "bowling_team_id": "T10",
        "batter_id": "P0140",
        "non_striker_id": "P0131",
        "bowler_id": "P0240",
        "runs_batter": 2,
        "runs_extras": 0,
        "total_runs": 2,
        "wicket_flag": False,
        "dismissed_player_id": None,
        "source_delivery_id": "DLV-M2022-001-1-02-02",
        "event_sequence": 8
    }
]

content = "\n".join(json.dumps(event) for event in drop_002)

dbutils.fs.put(
    input_path + "live_ball_event_drop_002.json",
    content,
    overwrite=True
)

display(dbutils.fs.ls(input_path))


Wrote 1433 bytes.


path,name,size,modificationTime
dbfs:/Volumes/ipl_matchday_360/default/ipl_matchday_data/streaming_input/live_ball_event_drop_001.json,live_ball_event_drop_001.json,2207,1791383498000
dbfs:/Volumes/ipl_matchday_360/default/ipl_matchday_data/streaming_input/live_ball_event_drop_002.json,live_ball_event_drop_002.json,1433,1791383700000


In [0]:
import json

input_path = "/Volumes/ipl_matchday_360/default/ipl_matchday_data/streaming_input/"

drop_003 = [
    {
        "event_id": "EVT-00009",
        "event_ts": "2026-07-20T19:03:00",
        "schema_version": 1.0,
        "match_id": "M2022-001",
        "innings_no": 1,
        "over_no": 2,
        "ball_seq": 3,
        "event_type": "scored_ball",
        "batting_team_id": "T06",
        "bowling_team_id": "T10",
        "batter_id": "P0131",
        "non_striker_id": "P0140",
        "bowler_id": "P0240",
        "runs_batter": 1,
        "runs_extras": 0,
        "total_runs": 1,
        "wicket_flag": False,
        "dismissed_player_id": None,
        "source_delivery_id": "DLV-M2022-001-1-02-03",
        "event_sequence": 9
    },
    
    # Duplicate event_id — used to test deduplication
    {
        "event_id": "EVT-00008",
        "event_ts": "2026-07-20T19:02:12",
        "schema_version": 1.0,
        "match_id": "M2022-001",
        "innings_no": 1,
        "over_no": 2,
        "ball_seq": 2,
        "event_type": "scored_ball",
        "batting_team_id": "T06",
        "bowling_team_id": "T10",
        "batter_id": "P0140",
        "non_striker_id": "P0131",
        "bowler_id": "P0240",
        "runs_batter": 2,
        "runs_extras": 0,
        "total_runs": 2,
        "wicket_flag": False,
        "dismissed_player_id": None,
        "source_delivery_id": "DLV-M2022-001-1-02-02",
        "event_sequence": 8
    },

    # Invalid run arithmetic — 2 + 1 should be 3, not 5
    {
        "event_id": "EVT-00010",
        "event_ts": "2026-07-20T19:03:12",
        "schema_version": 1.0,
        "match_id": "M2022-001",
        "innings_no": 1,
        "over_no": 2,
        "ball_seq": 4,
        "event_type": "scored_ball",
        "batting_team_id": "T06",
        "bowling_team_id": "T10",
        "batter_id": "P0131",
        "non_striker_id": "P0140",
        "bowler_id": "P0240",
        "runs_batter": 2,
        "runs_extras": 1,
        "total_runs": 5,
        "wicket_flag": False,
        "dismissed_player_id": None,
        "source_delivery_id": "DLV-M2022-001-1-02-04",
        "event_sequence": 10
    }
]

content = "\n".join(json.dumps(event) for event in drop_003)

dbutils.fs.put(
    input_path + "live_ball_event_drop_003.json",
    content,
    overwrite=True
)

display(dbutils.fs.ls(input_path))

Wrote 1437 bytes.


path,name,size,modificationTime
dbfs:/Volumes/ipl_matchday_360/default/ipl_matchday_data/streaming_input/live_ball_event_drop_001.json,live_ball_event_drop_001.json,2207,1791383498000
dbfs:/Volumes/ipl_matchday_360/default/ipl_matchday_data/streaming_input/live_ball_event_drop_002.json,live_ball_event_drop_002.json,1433,1791383700000
dbfs:/Volumes/ipl_matchday_360/default/ipl_matchday_data/streaming_input/live_ball_event_drop_003.json,live_ball_event_drop_003.json,1437,1791383735000


In [0]:
import json

input_path = "/Volumes/ipl_matchday_360/default/ipl_matchday_data/streaming_input/"

drop_004 = [
    {
        "event_id": "EVT-00011",
        "event_ts": "2026-07-20T19:04:00",
        "schema_version": 1.0,
        "match_id": "M2022-001",
        "innings_no": 1,
        "over_no": 2,
        "ball_seq": 5,
        "event_type": "scored_ball",
        "batting_team_id": "T06",
        "bowling_team_id": "T10",
        "batter_id": "P0131",
        "non_striker_id": "P0140",
        "bowler_id": "P0240",
        "runs_batter": 1,
        "runs_extras": 0,
        "total_runs": 1,
        "wicket_flag": False,
        "dismissed_player_id": None,
        "source_delivery_id": "DLV-M2022-001-1-02-05",
        "event_sequence": 11
    },
    {
        "event_id": "EVT-00012",
        "event_ts": "2026-07-20T19:04:12",
        "schema_version": 1.0,
        "match_id": "M2022-001",
        "innings_no": 1,
        "over_no": 2,
        "ball_seq": 6,
        "event_type": "wicket",
        "batting_team_id": "T06",
        "bowling_team_id": "T10",
        "batter_id": "P0131",
        "non_striker_id": "P0140",
        "bowler_id": "P0240",
        "runs_batter": 0,
        "runs_extras": 0,
        "total_runs": 0,
        "wicket_flag": True,
        "dismissed_player_id": "P0131",
        "source_delivery_id": "DLV-M2022-001-1-02-06",
        "event_sequence": 12
    }
]

content = "\n".join(json.dumps(event) for event in drop_004)

dbutils.fs.put(
    input_path + "live_ball_event_drop_004.json",
    content,
    overwrite=True
)

display(dbutils.fs.ls(input_path))

Wrote 956 bytes.


path,name,size,modificationTime
dbfs:/Volumes/ipl_matchday_360/default/ipl_matchday_data/streaming_input/live_ball_event_drop_001.json,live_ball_event_drop_001.json,2207,1791383498000
dbfs:/Volumes/ipl_matchday_360/default/ipl_matchday_data/streaming_input/live_ball_event_drop_002.json,live_ball_event_drop_002.json,1433,1791383700000
dbfs:/Volumes/ipl_matchday_360/default/ipl_matchday_data/streaming_input/live_ball_event_drop_003.json,live_ball_event_drop_003.json,1437,1791383735000
dbfs:/Volumes/ipl_matchday_360/default/ipl_matchday_data/streaming_input/live_ball_event_drop_004.json,live_ball_event_drop_004.json,956,1791383750000


In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, BooleanType,
    TimestampType
)

event_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("event_ts", TimestampType(), False),
    StructField("schema_version", DoubleType(), False),
    StructField("match_id", StringType(), False),
    StructField("innings_no", IntegerType(), False),
    StructField("over_no", IntegerType(), False),
    StructField("ball_seq", IntegerType(), False),
    StructField("event_type", StringType(), False),
    StructField("batting_team_id", StringType(), False),
    StructField("bowling_team_id", StringType(), False),
    StructField("batter_id", StringType(), False),
    StructField("non_striker_id", StringType(), False),
    StructField("bowler_id", StringType(), False),
    StructField("runs_batter", IntegerType(), False),
    StructField("runs_extras", IntegerType(), False),
    StructField("total_runs", IntegerType(), False),
    StructField("wicket_flag", BooleanType(), False),
    StructField("dismissed_player_id", StringType(), True),
    StructField("source_delivery_id", StringType(), False),
    StructField("event_sequence", IntegerType(), False)
])

print("20-field event schema created successfully")

20-field event schema created successfully


Configure the streaming input

In [0]:
streaming_input = "/Volumes/ipl_matchday_360/default/ipl_matchday_data/streaming_input/"

checkpoint_path = "/Volumes/ipl_matchday_360/default/ipl_matchday_data/checkpoints/live_ball_event/"

bronze_table = "workspace.default.bronze_live_ball_event"

In [0]:
print("Streaming input:", streaming_input)
print("Checkpoint:", checkpoint_path)
print("Bronze table:", bronze_table)

Streaming input: /Volumes/ipl_matchday_360/default/ipl_matchday_data/streaming_input/
Checkpoint: /Volumes/ipl_matchday_360/default/ipl_matchday_data/checkpoints/live_ball_event/
Bronze table: workspace.default.bronze_live_ball_event


Read the JSON files using Auto Loader

In [0]:
streaming_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .schema(event_schema)
    .load(streaming_input)
)

print("Auto Loader stream created successfully")

Auto Loader stream created successfully


Add watermark and deduplication

In [0]:
trusted_stream = (
    streaming_df
    .withWatermark("event_ts", "10 minutes")
    .dropDuplicates(["event_id"])
)

print("Watermark and event_id deduplication configured")

Watermark and event_id deduplication configured


Start the Bronze stream

In [0]:
bronze_table = "ipl_matchday_360.default.bronze_live_ball_event"

bronze_query = (
    trusted_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

print("Bronze streaming query started with AvailableNow")

Bronze streaming query started with AvailableNow


In [0]:
bronze_query.awaitTermination()
print("Bronze streaming batch completed")

Bronze streaming batch completed


In [0]:
display(
    spark.sql("""
        SELECT *
        FROM ipl_matchday_360.default.bronze_live_ball_event
        ORDER BY event_sequence
    """)
)

event_id,event_ts,schema_version,match_id,innings_no,over_no,ball_seq,event_type,batting_team_id,bowling_team_id,batter_id,non_striker_id,bowler_id,runs_batter,runs_extras,total_runs,wicket_flag,dismissed_player_id,source_delivery_id,event_sequence,_rescued_data
EVT-00001,2026-07-20T19:00:12.000Z,null,M2022-001,1,1,1,boundary,T06,T10,P0140,P0131,P0240,4,0,4,false,null,DLV-M2022-001-1-01-01,1,"{""schema_version"":""1.0"",""_file_path"":""/Volumes/ipl_matchday_360/default/ipl_matchday_data/streaming_input/live_ball_event_drop_001.json""}"
EVT-00002,2026-07-20T19:00:24.000Z,null,M2022-001,1,1,2,scored_ball,T06,T10,P0140,P0131,P0240,1,0,1,false,null,DLV-M2022-001-1-01-02,2,"{""schema_version"":""1.0"",""_file_path"":""/Volumes/ipl_matchday_360/default/ipl_matchday_data/streaming_input/live_ball_event_drop_001.json""}"
EVT-00003,2026-07-20T19:00:36.000Z,null,M2022-001,1,1,3,scored_ball,T06,T10,P0131,P0140,P0240,0,0,0,false,null,DLV-M2022-001-1-01-03,3,"{""schema_version"":""1.0"",""_file_path"":""/Volumes/ipl_matchday_360/default/ipl_matchday_data/streaming_input/live_ball_event_drop_001.json""}"
EVT-00004,2026-07-20T19:00:48.000Z,null,M2022-001,1,1,4,scored_ball,T06,T10,P0131,P0140,P0240,0,0,0,false,null,DLV-M2022-001-1-01-04,4,"{""schema_version"":""1.0"",""_file_path"":""/Volumes/ipl_matchday_360/default/ipl_matchday_data/streaming_input/live_ball_event_drop_001.json""}"
EVT-00005,2026-07-20T19:01:00.000Z,null,M2022-001,1,1,5,scored_ball,T06,T10,P0131,P0140,P0240,0,0,0,false,null,DLV-M2022-001-1-01-05,5,"{""schema_version"":""1.0"",""_file_path"":""/Volumes/ipl_matchday_360/default/ipl_matchday_data/streaming_input/live_ball_event_drop_001.json""}"
EVT-00006,2026-07-20T19:01:12.000Z,1.0,M2022-001,1,1,6,scored_ball,T06,T10,P0131,P0140,P0240,1,0,1,false,null,DLV-M2022-001-1-01-06,6,null
EVT-00007,2026-07-20T19:02:00.000Z,1.0,M2022-001,1,2,1,boundary,T06,T10,P0140,P0131,P0240,4,0,4,false,null,DLV-M2022-001-1-02-01,7,null
EVT-00008,2026-07-20T19:02:12.000Z,1.0,M2022-001,1,2,2,scored_ball,T06,T10,P0140,P0131,P0240,2,0,2,false,null,DLV-M2022-001-1-02-02,8,null
EVT-00009,2026-07-20T19:03:00.000Z,1.0,M2022-001,1,2,3,scored_ball,T06,T10,P0131,P0140,P0240,1,0,1,false,null,DLV-M2022-001-1-02-03,9,null
EVT-00010,2026-07-20T19:03:12.000Z,1.0,M2022-001,1,2,4,scored_ball,T06,T10,P0131,P0140,P0240,2,1,5,false,null,DLV-M2022-001-1-02-04,10,null


In [0]:
try:
    bronze_query.stop()
    print("Streaming query stopped")
except:
    print("No active query to stop")

Streaming query stopped


Check the Bronze table count

In [0]:
display(
    spark.sql("""
        SELECT
            COUNT(*) AS total_events,
            COUNT(DISTINCT event_id) AS unique_events
        FROM ipl_matchday_360.default.bronze_live_ball_event
    """)
)

total_events,unique_events
12,12


In [0]:
display(
    spark.sql("""
        SELECT event_id, COUNT(*) AS occurrence_count
        FROM ipl_matchday_360.default.bronze_live_ball_event
        GROUP BY event_id
        HAVING COUNT(*) > 1
    """)
)

event_id,occurrence_count


In [0]:
from pyspark.sql.functions import (
    col,
    when,
    lit
)

bronze_stream = (
    spark.readStream
    .table("ipl_matchday_360.default.bronze_live_ball_event")
)

print("Bronze streaming source created")

Bronze streaming source created


In [0]:
dq_stream = (
    bronze_stream
    .withColumn(
        "dq_reason",
        when(col("event_id").isNull(), lit("MISSING_EVENT_ID"))
        .when(col("match_id").isNull(), lit("MISSING_MATCH_ID"))
        .when(
            col("total_runs") != (col("runs_batter") + col("runs_extras")),
            lit("RUN_TOTAL_MISMATCH")
        )
        .when(
            (col("wicket_flag") == True) &
            col("dismissed_player_id").isNull(),
            lit("WICKET_PLAYER_MISSING")
        )
        .otherwise(lit(None))
    )
    .withColumn(
        "dq_status",
        when(col("dq_reason").isNull(), lit("VALID"))
        .otherwise(lit("QUARANTINE"))
    )
)

print("DQ rules applied")

DQ rules applied


In [0]:
display(
    dq_stream
    .select(
        "event_id",
        "event_sequence",
        "runs_batter",
        "runs_extras",
        "total_runs",
        "dq_status",
        "dq_reason"
    )
    .orderBy("event_sequence")
)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7772998145306672>, line 1
----> 1 display(
      2     dq_stream
      3     .select(
      4         "event_id",
      5         "event_sequence",
      6         "runs_batter",
      7         "runs_extras",
      8         "total_runs",
      9         "dq_status",
     10         "dq_reason"
     11     )
     12     .orderBy("event_sequence")
     13 )

File <command-7772998145306671>, line 2
      1 dq_stream = (
----> 2     bronze_stream
      3     .withColumn(
      4         "dq_reason",
      5         when(col("event_id").isNull(), lit("MISSING_EVENT_ID"))
      6         .when(col("match_id").isNull(), lit("MISSING_MATCH_ID"))
      7         .when(
      8             col("total_runs") != (col("runs_batter") + col("runs_extras")),
      9             lit("RUN_TOTAL_MISMATCH")
     10         )
     11        

In [0]:
from pyspark.sql.functions import col, when, lit

bronze_stream = (
    spark.readStream
    .table("ipl_matchday_360.default.bronze_live_ball_event")
)

dq_stream = (
    bronze_stream
    .withColumn(
        "dq_reason",
        when(col("event_id").isNull(), lit("MISSING_EVENT_ID"))
        .when(col("match_id").isNull(), lit("MISSING_MATCH_ID"))
        .when(
            col("total_runs") != (col("runs_batter") + col("runs_extras")),
            lit("RUN_TOTAL_MISMATCH")
        )
        .when(
            (col("wicket_flag") == True) &
            col("dismissed_player_id").isNull(),
            lit("WICKET_PLAYER_MISSING")
        )
        .otherwise(lit(None))
    )
    .withColumn(
        "dq_status",
        when(col("dq_reason").isNull(), lit("VALID"))
        .otherwise(lit("QUARANTINE"))
    )
)

In [0]:
from pyspark.sql.functions import col, when, lit

dq_check = (
    spark.table("ipl_matchday_360.default.bronze_live_ball_event")
    .withColumn(
        "dq_reason",
        when(col("event_id").isNull(), lit("MISSING_EVENT_ID"))
        .when(col("match_id").isNull(), lit("MISSING_MATCH_ID"))
        .when(
            col("total_runs") != (col("runs_batter") + col("runs_extras")),
            lit("RUN_TOTAL_MISMATCH")
        )
        .when(
            (col("wicket_flag") == True) &
            col("dismissed_player_id").isNull(),
            lit("WICKET_PLAYER_MISSING")
        )
        .otherwise(lit(None))
    )
    .withColumn(
        "dq_status",
        when(col("dq_reason").isNull(), lit("VALID"))
        .otherwise(lit("QUARANTINE"))
    )
)

display(
    dq_check.select(
        "event_id",
        "event_sequence",
        "runs_batter",
        "runs_extras",
        "total_runs",
        "dq_status",
        "dq_reason"
    ).orderBy("event_sequence")
)

event_id,event_sequence,runs_batter,runs_extras,total_runs,dq_status,dq_reason
EVT-00001,1,4,0,4,VALID,null
EVT-00002,2,1,0,1,VALID,null
EVT-00003,3,0,0,0,VALID,null
EVT-00004,4,0,0,0,VALID,null
EVT-00005,5,0,0,0,VALID,null
EVT-00006,6,1,0,1,VALID,null
EVT-00007,7,4,0,4,VALID,null
EVT-00008,8,2,0,2,VALID,null
EVT-00009,9,1,0,1,VALID,null
EVT-00010,10,2,1,5,QUARANTINE,RUN_TOTAL_MISMATCH


In [0]:
display(
    dq_check
    .filter(col("event_id") == "EVT-00010")
    .select(
        "event_id",
        "runs_batter",
        "runs_extras",
        "total_runs",
        "dq_status",
        "dq_reason"
    )
)

event_id,runs_batter,runs_extras,total_runs,dq_status,dq_reason
EVT-00010,2,1,5,QUARANTINE,RUN_TOTAL_MISMATCH


In [0]:
display(
    dq_check.groupBy("dq_status", "dq_reason")
    .count()
    .orderBy("dq_status")
)

dq_status,dq_reason,count
QUARANTINE,RUN_TOTAL_MISMATCH,1
VALID,null,11


Create the Trusted Silver table


In [0]:
trusted_table = "ipl_matchday_360.default.silver_live_ball_event"
trusted_checkpoint = "/Volumes/ipl_matchday_360/default/ipl_matchday_data/checkpoints/silver_live_ball_event/"

trusted_stream = (
    dq_stream
    .filter(col("dq_status") == "VALID")
    .drop("dq_reason", "dq_status")
)

trusted_query = (
    trusted_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", trusted_checkpoint)
    .trigger(availableNow=True)
    .toTable(trusted_table)
)

trusted_query.awaitTermination()

print("Trusted Silver table created successfully")

Trusted Silver table created successfully


In [0]:
display(
    spark.sql("""
        SELECT
            event_id,
            event_sequence,
            event_type,
            total_runs,
            wicket_flag
        FROM ipl_matchday_360.default.silver_live_ball_event
        ORDER BY event_sequence
    """)
)

event_id,event_sequence,event_type,total_runs,wicket_flag
EVT-00001,1,boundary,4,false
EVT-00002,2,scored_ball,1,false
EVT-00003,3,scored_ball,0,false
EVT-00004,4,scored_ball,0,false
EVT-00005,5,scored_ball,0,false
EVT-00006,6,scored_ball,1,false
EVT-00007,7,boundary,4,false
EVT-00008,8,scored_ball,2,false
EVT-00009,9,scored_ball,1,false
EVT-00011,11,scored_ball,1,false


Create Quarantine table

In [0]:
quarantine_table = "ipl_matchday_360.default.quarantine_live_ball_event"
quarantine_checkpoint = "/Volumes/ipl_matchday_360/default/ipl_matchday_data/checkpoints/quarantine_live_ball_event/"

quarantine_stream = (
    dq_stream
    .filter(col("dq_status") == "QUARANTINE")
)

quarantine_query = (
    quarantine_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", quarantine_checkpoint)
    .trigger(availableNow=True)
    .toTable(quarantine_table)
)

quarantine_query.awaitTermination()

print("Quarantine table created successfully")

Quarantine table created successfully


In [0]:
display(
    spark.sql("""
        SELECT
            event_id,
            event_sequence,
            runs_batter,
            runs_extras,
            total_runs,
            dq_status,
            dq_reason
        FROM ipl_matchday_360.default.quarantine_live_ball_event
        ORDER BY event_sequence
    """)
)

event_id,event_sequence,runs_batter,runs_extras,total_runs,dq_status,dq_reason
EVT-00010,10,2,1,5,QUARANTINE,RUN_TOTAL_MISMATCH


Create the live summary stream

In [0]:
from pyspark.sql.functions import sum, count, max

silver_stream = (
    spark.readStream
    .table("ipl_matchday_360.default.silver_live_ball_event")
)

live_health_summary = (
    silver_stream
    .groupBy(
        "match_id",
        "innings_no",
        "batting_team_id"
    )
    .agg(
        sum("total_runs").alias("current_runs"),
        count("*").alias("events_processed"),
        sum(
            when(col("wicket_flag") == True, 1).otherwise(0)
        ).alias("wickets")
    )
)

print("Live health summary stream created")

Live health summary stream created


Write it to a Gold table

In [0]:
live_summary_table = "ipl_matchday_360.default.gold_live_health_summary"

live_summary_checkpoint = (
    "/Volumes/ipl_matchday_360/default/"
    "ipl_matchday_data/checkpoints/gold_live_health_summary/"
)

live_summary_query = (
    live_health_summary.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", live_summary_checkpoint)
    .trigger(availableNow=True)
    .toTable(live_summary_table)
)

live_summary_query.awaitTermination()

print("Live health summary created successfully")

Live health summary created successfully


In [0]:
display(
    spark.sql("""
        SELECT *
        FROM ipl_matchday_360.default.gold_live_health_summary
        ORDER BY match_id, innings_no, batting_team_id
    """)
)

match_id,innings_no,batting_team_id,current_runs,events_processed,wickets
M2022-001,1,T06,14,11,1


In [0]:
display(
    spark.sql("""
        SELECT 'Bronze' AS layer, COUNT(*) AS records
        FROM ipl_matchday_360.default.bronze_live_ball_event

        UNION ALL

        SELECT 'Silver' AS layer, COUNT(*) AS records
        FROM ipl_matchday_360.default.silver_live_ball_event

        UNION ALL

        SELECT 'Quarantine' AS layer, COUNT(*) AS records
        FROM ipl_matchday_360.default.quarantine_live_ball_event
    """)
)

layer,records
Bronze,12
Silver,11
Quarantine,1


In [0]:
display(
    spark.sql("""
        SELECT event_id, dq_reason
        FROM ipl_matchday_360.default.quarantine_live_ball_event
        ORDER BY event_sequence
    """)
)

event_id,dq_reason
EVT-00010,RUN_TOTAL_MISMATCH


In [0]:
display(
    spark.sql("""
        SELECT event_id
        FROM ipl_matchday_360.default.silver_live_ball_event
        WHERE event_id = 'EVT-00010'
    """)
)

event_id


In [0]:
display(
    spark.sql("""
        SELECT
            event_id,
            COUNT(*) AS occurrence_count
        FROM ipl_matchday_360.default.bronze_live_ball_event
        GROUP BY event_id
        HAVING COUNT(*) > 1
    """)
)

event_id,occurrence_count


Final pipeline count

In [0]:
display(
    spark.sql("""
        SELECT 'Bronze' AS layer, COUNT(*) AS records
        FROM ipl_matchday_360.default.bronze_live_ball_event

        UNION ALL

        SELECT 'Silver' AS layer, COUNT(*) AS records
        FROM ipl_matchday_360.default.silver_live_ball_event

        UNION ALL

        SELECT 'Quarantine' AS layer, COUNT(*) AS records
        FROM ipl_matchday_360.default.quarantine_live_ball_event
    """)
)

layer,records
Bronze,12
Silver,11
Quarantine,1


In [0]:
display(
    spark.sql("""
        SELECT event_id, COUNT(*) AS count
        FROM ipl_matchday_360.default.bronze_live_ball_event
        GROUP BY event_id
        HAVING COUNT(*) > 1
    """)
)

event_id,count


In [0]:
display(
    spark.sql("""
        SELECT
            event_id,
            event_sequence,
            runs_batter,
            runs_extras,
            total_runs,
            dq_reason
        FROM ipl_matchday_360.default.quarantine_live_ball_event
        ORDER BY event_sequence
    """)
)

event_id,event_sequence,runs_batter,runs_extras,total_runs,dq_reason
EVT-00010,10,2,1,5,RUN_TOTAL_MISMATCH


In [0]:
display(
    spark.sql("""
        SELECT *
        FROM ipl_matchday_360.default.gold_live_health_summary
    """)
)

match_id,innings_no,batting_team_id,current_runs,events_processed,wickets
M2022-001,1,T06,14,11,1


In [0]:
live_summary_query_2 = (
    live_health_summary.writeStream
    .format("delta")
    .outputMode("complete")
    .option(
        "checkpointLocation",
        "/Volumes/ipl_matchday_360/default/ipl_matchday_data/checkpoints/gold_live_health_summary/"
    )
    .trigger(availableNow=True)
    .toTable("ipl_matchday_360.default.gold_live_health_summary")
)

live_summary_query_2.awaitTermination()

print("Checkpoint validation completed")

Checkpoint validation completed


In [0]:
display(
    spark.sql("""
        SELECT *
        FROM ipl_matchday_360.default.gold_live_health_summary
    """)
)

match_id,innings_no,batting_team_id,current_runs,events_processed,wickets
M2022-001,1,T06,14,11,1


verify the complete pipeline

In [0]:
display(
    spark.sql("""
        SELECT
            'Bronze' AS layer,
            COUNT(*) AS records
        FROM ipl_matchday_360.default.bronze_live_ball_event

        UNION ALL

        SELECT
            'Silver' AS layer,
            COUNT(*) AS records
        FROM ipl_matchday_360.default.silver_live_ball_event

        UNION ALL

        SELECT
            'Quarantine' AS layer,
            COUNT(*) AS records
        FROM ipl_matchday_360.default.quarantine_live_ball_event
    """)
)

layer,records
Bronze,12
Silver,11
Quarantine,1


In [0]:
display(
    spark.sql("""
        SELECT
            event_id,
            dq_reason
        FROM ipl_matchday_360.default.quarantine_live_ball_event
    """)
)

event_id,dq_reason
EVT-00010,RUN_TOTAL_MISMATCH


In [0]:
display(
    spark.sql("""
        SELECT 'Bronze' AS layer, COUNT(*) AS records
        FROM ipl_matchday_360.default.bronze_live_ball_event

        UNION ALL

        SELECT 'Silver' AS layer, COUNT(*) AS records
        FROM ipl_matchday_360.default.silver_live_ball_event

        UNION ALL

        SELECT 'Quarantine' AS layer, COUNT(*) AS records
        FROM ipl_matchday_360.default.quarantine_live_ball_event
    """)
)

layer,records
Bronze,12
Silver,11
Quarantine,1


verify the Quarantine record

In [0]:
display(
    spark.sql("""
        SELECT
            event_id,
            event_sequence,
            runs_batter,
            runs_extras,
            total_runs,
            dq_status,
            dq_reason
        FROM ipl_matchday_360.default.quarantine_live_ball_event
        ORDER BY event_sequence
    """)
)

event_id,event_sequence,runs_batter,runs_extras,total_runs,dq_status,dq_reason
EVT-00010,10,2,1,5,QUARANTINE,RUN_TOTAL_MISMATCH


verify Silver

In [0]:
display(
    spark.sql("""
        SELECT
            event_id,
            event_sequence,
            event_type,
            total_runs,
            wicket_flag
        FROM ipl_matchday_360.default.silver_live_ball_event
        ORDER BY event_sequence
    """)
)

event_id,event_sequence,event_type,total_runs,wicket_flag
EVT-00001,1,boundary,4,false
EVT-00002,2,scored_ball,1,false
EVT-00003,3,scored_ball,0,false
EVT-00004,4,scored_ball,0,false
EVT-00005,5,scored_ball,0,false
EVT-00006,6,scored_ball,1,false
EVT-00007,7,boundary,4,false
EVT-00008,8,scored_ball,2,false
EVT-00009,9,scored_ball,1,false
EVT-00011,11,scored_ball,1,false


Gold Live Health Summary.


In [0]:
display(
    spark.sql("""
        SELECT
            match_id,
            innings_no,
            batting_team_id,
            current_runs,
            events_processed,
            wickets
        FROM ipl_matchday_360.default.gold_live_health_summary
        ORDER BY match_id, innings_no
    """)
)

match_id,innings_no,batting_team_id,current_runs,events_processed,wickets
M2022-001,1,T06,14,11,1


duplicate-event validation

In [0]:
display(
    spark.sql("""
        SELECT
            event_id,
            COUNT(*) AS count
        FROM ipl_matchday_360.default.bronze_live_ball_event
        GROUP BY event_id
        HAVING COUNT(*) > 1
    """)
)

event_id,count


final Week 10 table validation.

In [0]:
display(
    spark.sql("""
        SELECT
            'Bronze' AS layer,
            COUNT(*) AS records
        FROM ipl_matchday_360.default.bronze_live_ball_event

        UNION ALL

        SELECT
            'Silver' AS layer,
            COUNT(*) AS records
        FROM ipl_matchday_360.default.silver_live_ball_event

        UNION ALL

        SELECT
            'Quarantine' AS layer,
            COUNT(*) AS records
        FROM ipl_matchday_360.default.quarantine_live_ball_event

        UNION ALL

        SELECT
            'Gold' AS layer,
            COUNT(*) AS records
        FROM ipl_matchday_360.default.gold_live_health_summary
    """)
)

layer,records
Bronze,12
Silver,11
Quarantine,1
Gold,1


In [0]:
from pyspark.sql.functions import col

display(
    spark.table("ipl_matchday_360.default.bronze_live_ball_event")
    .select(
        "event_id",
        "event_ts",
        "source_delivery_id",
        "event_sequence"
    )
    .orderBy("event_sequence")
)

event_id,event_ts,source_delivery_id,event_sequence
EVT-00001,2026-07-20T19:00:12.000Z,DLV-M2022-001-1-01-01,1
EVT-00002,2026-07-20T19:00:24.000Z,DLV-M2022-001-1-01-02,2
EVT-00003,2026-07-20T19:00:36.000Z,DLV-M2022-001-1-01-03,3
EVT-00004,2026-07-20T19:00:48.000Z,DLV-M2022-001-1-01-04,4
EVT-00005,2026-07-20T19:01:00.000Z,DLV-M2022-001-1-01-05,5
EVT-00006,2026-07-20T19:01:12.000Z,DLV-M2022-001-1-01-06,6
EVT-00007,2026-07-20T19:02:00.000Z,DLV-M2022-001-1-02-01,7
EVT-00008,2026-07-20T19:02:12.000Z,DLV-M2022-001-1-02-02,8
EVT-00009,2026-07-20T19:03:00.000Z,DLV-M2022-001-1-02-03,9
EVT-00010,2026-07-20T19:03:12.000Z,DLV-M2022-001-1-02-04,10


In [0]:
spark.readStream

In [0]:
.writeStream

  File <command-7772998145306716>, line 1
    .writeStream
    ^
SyntaxError: invalid syntax


In [0]:
spark.readStream

In [0]:
trusted_stream.writeStream

In [0]:
.writeStream

  File <command-7772998145306719>, line 1
    .writeStream
    ^
SyntaxError: invalid syntax


In [0]:
from pyspark.sql.functions import col

validation_stream = (
    spark.readStream
    .table("ipl_matchday_360.default.bronze_live_ball_event")
    .withWatermark("event_ts", "10 minutes")
    .dropDuplicates(["event_id"])
)

validation_query = (
    validation_stream
    .writeStream
    .format("console")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "/Volumes/ipl_matchday_360/default/ipl_matchday_data/checkpoints/week10_validation/"
    )
    .trigger(availableNow=True)
    .start()
)

validation_query.awaitTermination()

In [0]:
from pyspark.sql.functions import col

validation_stream = (
    spark.readStream
    .table("ipl_matchday_360.default.bronze_live_ball_event")
    .withWatermark("event_ts", "10 minutes")
    .dropDuplicates(["event_id"])
)

validation_query = (
    validation_stream
    .writeStream
    .format("console")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "/Volumes/ipl_matchday_360/default/ipl_matchday_data/checkpoints/week10_validation/"
    )
    .trigger(availableNow=True)
    .start()
)

validation_query.awaitTermination()

In [0]:
display(
    spark.table("ipl_matchday_360.default.bronze_live_ball_event")
    .groupBy("event_id")
    .count()
    .filter("count > 1")
)

event_id,count


In [0]:
from pyspark.sql.functions import col

display(
    spark.table("ipl_matchday_360.default.bronze_live_ball_event")
    .groupBy("source_delivery_id")
    .count()
    .filter(col("count") > 1)
)

source_delivery_id,count


Create a small watermark validation query

In [0]:
from pyspark.sql.functions import col

watermark_validation = (
    spark.readStream
    .table("ipl_matchday_360.default.bronze_live_ball_event")
    .withWatermark("event_ts", "10 minutes")
    .dropDuplicates(["event_id"])
)

watermark_query = (
    watermark_validation
    .writeStream
    .format("console")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "/Volumes/ipl_matchday_360/default/ipl_matchday_data/checkpoints/week10_watermark_validation/"
    )
    .trigger(availableNow=True)
    .start()
)

watermark_query.awaitTermination()

In [0]:

before_count = spark.table(
    "ipl_matchday_360.default.bronze_live_ball_event"
).count()

print("Records before rerun:", before_count)

rerun_stream = (
    spark.readStream
    .table("ipl_matchday_360.default.bronze_live_ball_event")
)

rerun_query = (
    rerun_stream
    .writeStream
    .format("console")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "/Volumes/ipl_matchday_360/default/ipl_matchday_data/checkpoints/week10_watermark_validation/"
    )
    .trigger(availableNow=True)
    .start()
)

rerun_query.awaitTermination()

after_count = spark.table(
    "ipl_matchday_360.default.bronze_live_ball_event"
).count()

print("Records after rerun:", after_count)
print("New records:", after_count - before_count)

Records before rerun: 12
Records after rerun: 12
New records: 0
